# 01 - Data Preparation

This notebook prepares a real, manually downloaded flood dataset. It does not create sample data. Start by searching for these public dataset leads; inspect the actual CSV and its data dictionary because none is guaranteed to contain every requested predictor.

1. **Kaggle tabular flood prediction datasets** - search: `flood prediction rainfall soil moisture elevation NDVI flood label CSV`. Prefer a CSV with an explicit binary flood/no-flood outcome and documented units.
2. **Sen1Floods11** - search: `Sen1Floods11 flood dataset labels download`. This public satellite benchmark provides flood labels and geospatial imagery, but environmental columns such as soil moisture and elevation may need to come from documented supplementary sources.
3. **NASA/Google Global Flood Database** - search: `NASA Global Flood Database 913 flood events download`. It provides flood-event information and mapped flood extent; it is not a ready-made table of all requested environmental predictors.

For each result, check the CSV row count (prefer thousands of observations and enough examples in both classes), whether a clearly defined binary flood label exists, and whether rainfall, soil moisture, elevation, and NDVI/vegetation are present. Also check units, dates, coordinates, license, and whether rows represent comparable locations/times. NDWI, humidity, slope, and temperature may require supplementary data. Do not combine sources until locations and dates can be aligned.

Download the original CSV file(s) yourself into `data/raw/`, preserve the originals, and update the CONFIG cell if the main file is not named `flood_data.csv`.

In [9]:
# Configure project paths and dataset-specific column mapping here so later cells use one consistent setup.
from pathlib import Path

PROJECT_ROOT_OVERRIDE = None
PROJECT_MARKERS = ("data", "models", "notebooks")


def find_project_root():
    if PROJECT_ROOT_OVERRIDE is not None:
        candidate = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        if all((candidate / marker).is_dir() for marker in PROJECT_MARKERS):
            return candidate
        raise FileNotFoundError(
            f"PROJECT_ROOT_OVERRIDE does not contain the project folders: {candidate}"
        )

    start_path = Path.cwd().resolve()
    for candidate in (start_path, *start_path.parents):
        if all((candidate / marker).is_dir() for marker in PROJECT_MARKERS):
            return candidate
    raise FileNotFoundError(
        f"Project root not found from kernel directory {start_path}. "
        "Set PROJECT_ROOT_OVERRIDE to a project path visible to this kernel. "
        "A remote kernel needs the project files mounted or copied into its filesystem."
    )


PROJECT_ROOT = find_project_root()
RAW_FILE_PATH = PROJECT_ROOT / "data" / "raw" / "flood_data.csv"
PROCESSED_FILE_PATH = PROJECT_ROOT / "data" / "processed" / "flood_data_processed.csv"
SCALER_PATH = PROJECT_ROOT / "models" / "scaler.joblib"
FEATURE_COLUMNS_PATH = PROJECT_ROOT / "models" / "feature_columns.json"

# Map source CSV headers to the canonical names below; leave empty entries out.
RENAME_MAP = {
    # Example: "Rainfall (mm)": "rainfall",
}
BASE_FEATURE_COLUMNS = [
    "rainfall",
    "soil_moisture",
    "humidity",
    "ndvi",
    "ndwi",
    "elevation",
    "slope",
    "temperature",
]
TARGET_COLUMN = "flood_label"
MODEL_FEATURE_COLUMNS = BASE_FEATURE_COLUMNS + ["cluster_label"]
MOSTLY_EMPTY_THRESHOLD = 0.50
FLOOD_LABEL_MAP = {
    "flood": 1, "yes": 1, "true": 1, "1": 1,
    "no flood": 0, "non-flood": 0, "no": 0, "false": 0, "0": 0,
}

FileNotFoundError: Project root not found from kernel directory /content. Set PROJECT_ROOT_OVERRIDE to a project path visible to this kernel. A remote kernel needs the project files mounted or copied into its filesystem.

## Load and Inspect the Source

This first inspection checks the downloaded file before any column changes or cleaning, so schema and data-quality issues stay visible.

In [ ]:
# Load the author's CSV and inspect its dimensions, types, missingness, and descriptive statistics before processing.
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

if not RAW_FILE_PATH.is_file():
    raise FileNotFoundError(
        f"CSV not found at {RAW_FILE_PATH}. Download a real dataset and place it there, or update RAW_FILE_PATH in CONFIG."
    )

raw_df = pd.read_csv(RAW_FILE_PATH)
print(f"Source file: {RAW_FILE_PATH.name}")
print(f"Shape: {raw_df.shape[0]} rows x {raw_df.shape[1]} columns")
print("\nColumn data types:")
display(raw_df.dtypes.to_frame(name="dtype"))
print("\nMissing values by column:")
display(raw_df.isna().sum().to_frame(name="missing_count"))
print("\nBasic statistics:")
display(raw_df.describe(include="all").transpose())

FileNotFoundError: CSV not found at /content/data/raw/flood_data.csv. Download a real dataset and place it there, or update RAW_FILE_PATH in CONFIG.

## Match the Required Schema

Edit `RENAME_MAP` in CONFIG to map source headers to the canonical names. This notebook requires all eight environmental predictors plus a binary `flood_label`. If fields are missing, it reports their exact names and stops; use a documented supplementary dataset or formally revise the feature contract rather than inventing values.

In [ ]:
# Standardize source headers, select only required columns, and verify the flood target is truly binary.
df = raw_df.rename(columns=RENAME_MAP).copy()
required_columns = BASE_FEATURE_COLUMNS + [TARGET_COLUMN]
missing_columns = [column for column in required_columns if column not in df.columns]

if missing_columns:
    print("Missing required columns:")
    for column in missing_columns:
        print(f"- {column}")
    print("\nUse a documented supplementary source for genuinely absent measurements, or formally drop a feature and update the shared training/app feature contract. No values have been fabricated.")
    raise ValueError("Dataset schema does not meet the required flood project feature contract.")

df = df[required_columns].copy()
for column in BASE_FEATURE_COLUMNS:
    df[column] = pd.to_numeric(df[column], errors="coerce")

original_labels = df[TARGET_COLUMN].copy()
normalized_labels = original_labels.astype("string").str.strip().str.lower()
normalized_labels = normalized_labels.replace("", pd.NA)
mapped_labels = normalized_labels.map(FLOOD_LABEL_MAP)
numeric_labels = pd.to_numeric(original_labels, errors="coerce")
df[TARGET_COLUMN] = mapped_labels.combine_first(numeric_labels)
unknown_label_mask = original_labels.notna() & normalized_labels.notna() & df[TARGET_COLUMN].isna()
if unknown_label_mask.any():
    unknown_values = original_labels[unknown_label_mask].astype(str).unique().tolist()
    raise ValueError(f"Unrecognized flood_label values: {unknown_values}. Map the source labels to 0 (no flood) and 1 (flood) in CONFIG.")

print("Selected columns:", required_columns)
print("Target values before missing-target removal:", sorted(df[TARGET_COLUMN].dropna().unique().tolist()))

## Remove Duplicates and Handle Missing Values

Exact duplicate observations are removed. Numeric predictors with some missing values are median-imputed. If a predictor is more than 50% empty, rows missing that predictor are removed and the row loss is reported; missing target labels are dropped because a class cannot be imputed. Fully empty predictors stop processing and require a supplementary source or a formally revised feature contract.

In [ ]:
# Clean duplicate and incomplete observations using explicit rules, while recording every affected row and feature.
rows_before_cleaning = len(df)
df = df.drop_duplicates().copy()
duplicates_removed = rows_before_cleaning - len(df)

all_empty_features = [column for column in BASE_FEATURE_COLUMNS if df[column].isna().all()]
if all_empty_features:
    raise ValueError(f"These predictors have no usable numeric values: {all_empty_features}. Check RENAME_MAP/source units or obtain a supplementary dataset.")

missing_fraction = df[BASE_FEATURE_COLUMNS].isna().mean()
mostly_empty_features = missing_fraction[missing_fraction > MOSTLY_EMPTY_THRESHOLD].index.tolist()
rows_before_sparse_filter = len(df)
if mostly_empty_features:
    print(f"Predictors more than {MOSTLY_EMPTY_THRESHOLD:.0%} missing: {mostly_empty_features}")
    df = df.dropna(subset=mostly_empty_features).copy()
rows_removed_for_sparse_features = rows_before_sparse_filter - len(df)
rows_before_missing_target = len(df)
df = df.dropna(subset=[TARGET_COLUMN]).copy()
rows_removed_for_missing_target = rows_before_missing_target - len(df)

imputed_counts = {}
for column in BASE_FEATURE_COLUMNS:
    median_value = df[column].median()
    if pd.isna(median_value):
        raise ValueError(f"Predictor {column!r} has no usable values after filtering.")
    imputed_counts[column] = int(df[column].isna().sum())
    df[column] = df[column].fillna(median_value)

df[TARGET_COLUMN] = df[TARGET_COLUMN].astype(int)
target_values = set(df[TARGET_COLUMN].unique())
if not target_values.issubset({0, 1}):
    raise ValueError(f"Flood label must contain only 0 and 1 after mapping; found {sorted(target_values)}.")
if len(target_values) < 2:
    raise ValueError("Only one flood_label class remains after cleaning; obtain more representative labeled data.")
if df.empty:
    raise ValueError("No rows remain after cleaning. Review the source dataset and missing-value rules.")

print(f"Duplicate rows removed: {duplicates_removed}")
print(f"Rows removed for missing mostly-empty predictors: {rows_removed_for_sparse_features}")
print(f"Rows removed for missing flood labels: {rows_removed_for_missing_target}")
print("Missing predictor values median-imputed:", imputed_counts)
print(f"Rows remaining: {len(df)}")

## Scale Predictors and Save Artifacts

The scaler is fit on environmental predictors only; `flood_label` is never scaled. The processed CSV retains the binary label. `cluster_label` is not available until notebook 02, but remains in the shared model feature order.

> **Leakage note:** This preparation stage fits the requested scaler on all cleaned rows. Before final evaluation/deployment, notebook 03 must split the data first and refit the scaler using only the training partition, then save that training-only scaler to the same configured path.

In [ ]:
# Scale only the eight measured predictors, then save the processed table, scaler, and exact future model input order.
scaler = StandardScaler()
scaled_values = scaler.fit_transform(df[BASE_FEATURE_COLUMNS])
processed_df = pd.DataFrame(scaled_values, columns=BASE_FEATURE_COLUMNS, index=df.index)
processed_df[TARGET_COLUMN] = df[TARGET_COLUMN].astype(int)

PROCESSED_FILE_PATH.parent.mkdir(parents=True, exist_ok=True)
SCALER_PATH.parent.mkdir(parents=True, exist_ok=True)
FEATURE_COLUMNS_PATH.parent.mkdir(parents=True, exist_ok=True)
processed_df.to_csv(PROCESSED_FILE_PATH, index=False)
joblib.dump(scaler, SCALER_PATH)
with FEATURE_COLUMNS_PATH.open("w", encoding="utf-8") as feature_file:
    json.dump(MODEL_FEATURE_COLUMNS, feature_file, indent=2)

print(f"Saved processed CSV: {PROCESSED_FILE_PATH}")
print(f"Saved scaler: {SCALER_PATH}")
print(f"Saved model feature order: {FEATURE_COLUMNS_PATH}")
print("Processed columns:", processed_df.columns.tolist())
print("Model features (cluster_label is added in notebook 02):", MODEL_FEATURE_COLUMNS)

## Data Preparation Summary

Review this summary after running the notebook with your chosen dataset. A large imputation count, substantial row loss, or weak class balance may require better source data before clustering and classification.

In [ ]:
# Summarize the final prepared data and its class balance so dataset limitations are visible before modeling.
class_counts = processed_df[TARGET_COLUMN].value_counts().reindex([0, 1], fill_value=0)
class_percentages = class_counts / len(processed_df) * 100

print(f"Final row count: {len(processed_df)}")
print(f"Final processed shape: {processed_df.shape}")
print(f"Non-flood (0): {class_counts.loc[0]} rows ({class_percentages.loc[0]:.2f}%)")
print(f"Flood (1): {class_counts.loc[1]} rows ({class_percentages.loc[1]:.2f}%)")
print(f"Duplicate rows removed: {duplicates_removed}")
print(f"Rows removed for sparse predictors: {rows_removed_for_sparse_features}")
print(f"Rows removed for missing labels: {rows_removed_for_missing_target}")
print("Predictors over the 50% missingness threshold:", mostly_empty_features)
print("Median-imputed predictor counts:", imputed_counts)
print("Data-quality notes: verify label meaning, units, geographic/temporal alignment, class balance, license, and the train-only scaler refit required in notebook 03.")